In [2]:
import numpy as np
import pandas as pd
works = pd.DataFrame({
'작가': ['김유정', '김유정', '김유정',
'현진건', '현진건',
'이상', '이상',
'나도향', '나도향',
'채만식', '이효석'],
'작품': ['봄봄', '동백꽃', '만무방',
'운수 좋은 날', 'B사감과 러브레터',
'날개', '권태',
'벙어리 삼룡이', '물레방아',
'레디메이드 인생', '메밀꽃 필 무렵'],
'연도': [1935, 1936, 1934,
1924, 1925,
1936, 1937,
1925, 1925,
1934, np.nan],
# 이효석 '메밀꽃 필 무렵' 발표연도가 누락됨
'글자수': [12500, 9800, 13600,
11200, 7400,
18300, 14100,
16800, 14500,
22000, 11800],
})
authors = pd.DataFrame({
'작가': ['김유정', '현진건', '이상', '나도향', '채만식', '이효석', '염상섭'],
'생년': [1908, 1900, 1910, 1902, 1902, 1907, 1897],
'몰년': [1937, 1943, 1937, 1926, 1950, 1942, 1963],
})

In [4]:
print(works.shape)
print(works.dtypes)
print(works.describe())

(11, 4)
작가         str
작품         str
연도     float64
글자수      int64
dtype: object
                연도           글자수
count    10.000000     11.000000
mean   1931.100000  13818.181818
std       5.546771   4080.641661
min    1924.000000   7400.000000
25%    1925.000000  11500.000000
50%    1934.000000  13600.000000
75%    1935.750000  15650.000000
max    1937.000000  22000.000000


연도 열에는 결측치(NaN)가 하나 포함되어 있는데
NaN은 부동소수점으로만 표현되므로 연도 열을 float64로 저장하고 결측치가 없는 글자수 열은 int64로 유지되기 때문

In [5]:
print(works.isna().sum())

작가     0
작품     0
연도     1
글자수    0
dtype: int64


In [6]:
works2 = works.copy()
works2['연도'] = works2['연도'].fillna(1936).astype(int)

print(works2.dtypes)
print(works2.tail(3))

작가       str
작품       str
연도     int64
글자수    int64
dtype: object
     작가        작품    연도    글자수
8   나도향      물레방아  1925  14500
9   채만식  레디메이드 인생  1934  22000
10  이효석  메밀꽃 필 무렵  1936  11800


fillna(1936)으로 누락된 연도를 1936으로 채운 뒤 NaN이 사라졌으므로 astype(int)로 정수형 변환이 가능
dtypes를 보면 연도가 int64로 바뀌고 tail(3)에서 마지막 행의 연도가 1936으로 채워진 것을 확인할 수 있음

In [7]:
after_1930 = works2[works2['연도'] >= 1930]
print(after_1930)

kim_or_lee = works2[works2['작가'].isin(['김유정', '이상'])]
print(kim_or_lee)

     작가        작품    연도    글자수
0   김유정        봄봄  1935  12500
1   김유정       동백꽃  1936   9800
2   김유정       만무방  1934  13600
5    이상        날개  1936  18300
6    이상        권태  1937  14100
9   채만식  레디메이드 인생  1934  22000
10  이효석  메밀꽃 필 무렵  1936  11800
    작가   작품    연도    글자수
0  김유정   봄봄  1935  12500
1  김유정  동백꽃  1936   9800
2  김유정  만무방  1934  13600
5   이상   날개  1936  18300
6   이상   권태  1937  14100


In [8]:
def categorize(n: int) -> str:
    if n < 10000:
        return '짧음'
    elif n <= 15000:   # 10000 <= n <= 15000
        return '보통'
    else:
        return '긴'

works2['분량'] = works2['글자수'].apply(categorize)
print(works2[['작품', '글자수', '분량']])

           작품    글자수  분량
0          봄봄  12500  보통
1         동백꽃   9800  짧음
2         만무방  13600  보통
3     운수 좋은 날  11200  보통
4   B사감과 러브레터   7400  짧음
5          날개  18300   긴
6          권태  14100  보통
7     벙어리 삼룡이  16800   긴
8        물레방아  14500  보통
9    레디메이드 인생  22000   긴
10   메밀꽃 필 무렵  11800  보통


In [9]:
sorted_works = works2.sort_values(by=['연도', '글자수'], ascending=[True, False])
print(sorted_works)

     작가         작품    연도    글자수  분량
3   현진건    운수 좋은 날  1924  11200  보통
7   나도향    벙어리 삼룡이  1925  16800   긴
8   나도향       물레방아  1925  14500  보통
4   현진건  B사감과 러브레터  1925   7400  짧음
9   채만식   레디메이드 인생  1934  22000   긴
2   김유정        만무방  1934  13600  보통
0   김유정         봄봄  1935  12500  보통
5    이상         날개  1936  18300   긴
10  이효석   메밀꽃 필 무렵  1936  11800  보통
1   김유정        동백꽃  1936   9800  짧음
6    이상         권태  1937  14100  보통


불 인덱싱과 isin으로 원하는 행을 필터링하고 apply로 글자수에 따라 분량 범주를 분류하는 새 열을 추가
마지막으로 sort_values에 기준을 리스트로 넘겨 다중 정렬

In [10]:
by_author = works2.groupby('작가')['글자수'].agg(['mean', 'count', 'sum'])
print(by_author)

             mean  count    sum
작가                             
김유정  11966.666667      3  35900
나도향  15650.000000      2  31300
이상   16200.000000      2  32400
이효석  11800.000000      1  11800
채만식  22000.000000      1  22000
현진건   9300.000000      2  18600


In [11]:
print(works2['분량'].value_counts())

분량
보통    6
긴     3
짧음    2
Name: count, dtype: int64


In [12]:
print(works2.groupby(['작가', '분량']).size())

작가   분량
김유정  보통    2
     짧음    1
나도향  긴     1
     보통    1
이상   긴     1
     보통    1
이효석  보통    1
채만식  긴     1
현진건  보통    1
     짧음    1
dtype: int64


작가별로 agg를 써서 평균·작품 수·총 글자수를 한 번에 계산하고, value_counts로 분량별 빈도
groupby([]).size()로 작가와 분량의 조합별 작품 수

In [13]:
full = pd.merge(works2, authors, on='작가', how='left')
print(full)
print(full.shape)

     작가         작품    연도    글자수  분량    생년    몰년
0   김유정         봄봄  1935  12500  보통  1908  1937
1   김유정        동백꽃  1936   9800  짧음  1908  1937
2   김유정        만무방  1934  13600  보통  1908  1937
3   현진건    운수 좋은 날  1924  11200  보통  1900  1943
4   현진건  B사감과 러브레터  1925   7400  짧음  1900  1943
5    이상         날개  1936  18300   긴  1910  1937
6    이상         권태  1937  14100  보통  1910  1937
7   나도향    벙어리 삼룡이  1925  16800   긴  1902  1926
8   나도향       물레방아  1925  14500  보통  1902  1926
9   채만식   레디메이드 인생  1934  22000   긴  1902  1950
10  이효석   메밀꽃 필 무렵  1936  11800  보통  1907  1942
(11, 7)


In [14]:
full['집필 당시 나이'] = full['연도'] - full['생년']
print(full[['작가', '작품', '연도', '생년', '집필 당시 나이']])

     작가         작품    연도    생년  집필 당시 나이
0   김유정         봄봄  1935  1908        27
1   김유정        동백꽃  1936  1908        28
2   김유정        만무방  1934  1908        26
3   현진건    운수 좋은 날  1924  1900        24
4   현진건  B사감과 러브레터  1925  1900        25
5    이상         날개  1936  1910        26
6    이상         권태  1937  1910        27
7   나도향    벙어리 삼룡이  1925  1902        23
8   나도향       물레방아  1925  1902        23
9   채만식   레디메이드 인생  1934  1902        32
10  이효석   메밀꽃 필 무렵  1936  1907        29


In [15]:
avg_age = full.groupby('작가')['집필 당시 나이'].mean().sort_values()
print(avg_age)

작가
나도향    23.0
현진건    24.5
이상     26.5
김유정    27.0
이효석    29.0
채만식    32.0
Name: 집필 당시 나이, dtype: float64


how='left'로 works2 기준 병합으로 작가 정보를 붙이고 집필 당시 나이를 구함
groupby와 sort_values를 이어 작가별 평균 집필 나이를 정렬해 비교